In [1]:
import json
import os

parent_dir = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
dataset_dir = f'{parent_dir}/dataset'

with open(f'{dataset_dir}/correct_ids/artists.json') as f:
    artists = json.load(f)

In [2]:
artists[0].keys()

dict_keys(['id_author', 'name', 'gender', 'birth_date', 'birth_place', 'nationality', 'description', 'active_start', 'active_end', 'province', 'region', 'country', 'latitude', 'longitude', 'type', 'active-end', 'new_id_artist'])

In [3]:
regions = set()

for artist in artists:
    if artist.get('region'):
        regions.add(artist['region'])

print(regions)

{'Sicilia', 'Québec', 'Lombardia', 'Liguria', 'Veneto', 'Andalucía', 'Sardegna', 'Toscana', 'Distrito Nacional', 'Campania', 'Ciudad Autónoma de Buenos Aires', 'Lazio', 'Calabria', 'Marche', 'Κύπρος', 'Sardigna/Sardegna', 'Piemonte', 'Puglia'}


In [4]:
not_italy = [
    'Andalucía', 'Québec', 'Distrito Nacional',
    'Κύπρος', 'Ciudad Autónoma de Buenos Aires'
]

outlier_artists = []

for artist in artists:
    if artist.get('region') and artist['region'] in not_italy:
        outlier_artists.append(artist)

len(outlier_artists)

5

In [5]:
with open(f'{dataset_dir}/artists.json') as f:
    og_artists = json.load(f)

In [6]:
for artist in og_artists:
    for outlier_artist in outlier_artists:
        if artist['name'] == outlier_artist['name']:
            print(artist)
            break

{'id_author': 'ART67409252', 'name': 'chadia rodriguez', 'gender': 'F', 'birth_date': '1998-11-07', 'birth_place': 'Almería', 'nationality': 'Italia', 'description': 'rapper italiana (1998-)', 'active_start': None, 'active_end': None, 'province': 'Genova', 'region': 'Liguria', 'country': 'Italia', 'latitude': 44.4194958, 'longitude': 8.9234821}
{'id_author': 'ART02733420', 'name': 'marracash', 'gender': 'M', 'birth_date': '1979-05-22', 'birth_place': 'Nicosia', 'nationality': 'Italia', 'description': 'rapper e produttore discografico italiano (1979-)', 'active_start': '1999-01-01', 'active_end': None, 'province': 'Enna', 'region': 'Sicilia', 'country': 'Italia', 'latitude': 37.747452, 'longitude': 14.397271}
{'id_author': 'ART87389753', 'name': 'priestess', 'gender': 'F', 'birth_date': None, 'birth_place': None, 'nationality': None, 'description': 'gruppo musicale canadese', 'active_start': '2003-01-01', 'active_end': None, 'province': None, 'region': None, 'country': None, 'latitude':

In [7]:
from fill import geo_query

geo_query('Genova')

[{'place_id': 419141609,
  'licence': 'Data © OpenStreetMap contributors, ODbL 1.0. http://osm.org/copyright',
  'osm_type': 'relation',
  'osm_id': 44875,
  'lat': '44.4072600',
  'lon': '8.9338624',
  'class': 'boundary',
  'type': 'administrative',
  'place_rank': 16,
  'importance': 0.7301929549030833,
  'addresstype': 'city',
  'name': 'Genova',
  'display_name': 'Genova, Liguria, Italia',
  'address': {'city': 'Genova',
   'county': 'Genova',
   'ISO3166-2-lvl6': 'IT-GE',
   'state': 'Liguria',
   'ISO3166-2-lvl4': 'IT-42',
   'country': 'Italia',
   'country_code': 'it'},
  'boundingbox': ['44.3784709', '44.5198419', '8.6657444', '9.0955805']}]

In [8]:
for artist in og_artists:
    for new_artist in artists:
        ans = None
        while not ans:
            try:
                if (artist['name'] ==  new_artist['name'] and 
                    artist.get('region')
                ):
                    if artist.get('province'):
                        new_artist['province'] = artist['province']
                        
                    new_artist['region'] = artist['region']

                    if artist.get('country'):
                        new_artist['country'] = artist['country']
                    else:
                        ans = geo_query(artist['region'])[0]
                        new_artist['country'] = ans['address']['country']
                    break
                elif (
                    artist['name'] == new_artist['name'] and
                    not artist.get('region')
                ):
                    if artist.get('province'):
                        new_artist['province'] = artist['province']
                        ans = geo_query(artist['province'])[0]
                        new_artist['region'] = ans['address']['state']
                        
                    if artist.get('country'):
                        new_artist['country'] = artist['country']

                    elif ans:
                        new_artist['country'] = ans['address']['country']
                    break
                break
            except Exception as e:
                print(e)

In [9]:
import requests

url = "https://nominatim.openstreetmap.org/reverse"

def rev_geo_query(lat, lon):
    params = {
        "lat": lat,
        "lon": lon,
        "format": "json",
        "addressdetails": 1
    }
    headers = {
        "User-Agent": "MyResearchProject/1.0 (b.barbieri7@studenti.unipi.it)"
    }
    response = requests.get(url, params=params, headers=headers)
    return response.json()

In [10]:
geo = []

for i in range(len(artists)):
    if (og_artists[i].get('latitude') and
        og_artists[i].get('longitude')
    ):
        ans = None
        while not ans:
            try:
                ans = rev_geo_query(
                    og_artists[i]['latitude'],
                    og_artists[i]['longitude']
                )
                if ans['address'].get('state'):
                    artists[i]['province'] = ans['address']['county']
                    artists[i]['region'] = ans['address']['state']
                    artists[i]['country'] = ans['address']['country']

                    flag = False
                    for place in geo:
                        if (
                            place['province'] == artists[i]['province'] and
                            place['region'] == artists[i]['region'] and
                            place['country'] == artists[i]['country']
                        ):
                            flag = True
                            artists[i]['latitude'] = place['latitude']
                            artists[i]['longitude'] = place['longitude']
                            break
                
                if not flag:
                    ans_place = geo_query(ans['address']['county'])[0]
                    artists[i]['latitude'] = ans_place['lat']
                    artists[i]['longitude'] = ans_place['lon']
                    geo.append({
                        # 'city': ans_place['address']['city'],
                        'province': ans_place['address']['county'],
                        'region': ans_place['address']['state'],
                        'country': ans_place['address']['country'],
                        'latitude': ans_place['lat'],
                        'longitude': ans_place['lon'],
                    })
                    
            except Exception as e:
                print(e)
                print(ans)

In [11]:
for place in geo:
    print(place['country'])

Italia
Italia
Italia
Italia
Italia
Italia
Italia
Italia
Italia
Italia
Italia
Italia
Italia
Italia
Italia
Italia
Italia
Italia
Italia
Italia
Italia
Italia
Italia
Italia
Italia
Italia


In [12]:
map_regions = {
    'Sardigna/Sardegna': 'Sardegna'
}

map_provinces = {
    'Casteddu/Cagliari': 'Cagliari'
}

for place in geo:
    if place['province'] in map_provinces.keys():
        place['province'] = map_provinces[place['province']]
    
    if place['region'] in map_regions.keys():
        place['region'] = map_regions[place['region']]

for artist in artists:
    if artist['province'] in map_provinces.keys():
        artist['province'] = map_provinces[artist['province']]
    
    if place['region'] in map_regions.keys():
        artist['region'] = map_regions[artist['region']]

In [13]:
from uuid import uuid4

new_ids = set()

for artist in artists:
    for place in geo:
        if not place.get('geo_id'):
            new_id = str(uuid4())
            while new_id in new_ids:
                new_id = str(uuid4())
            new_ids.add(new_id)
            place['geo_id'] = new_id
        
        if (place['country'] == artist['country'] and
            place['region'] == artist['region'] and
            place['province'] == artist['province']
        ):
            artist['geo_id'] = place['geo_id']

In [14]:
artists[0]

{'id_author': 'ART82291002',
 'name': '99 posse',
 'gender': 'M',
 'birth_date': None,
 'birth_place': 'Napoli',
 'nationality': None,
 'description': 'gruppo musicale italiano',
 'active_start': '1991-10-09',
 'active_end': None,
 'province': 'Napoli',
 'region': 'Campania',
 'country': 'Italia',
 'latitude': '40.8358846',
 'longitude': '14.2487679',
 'type': 'Group',
 'active-end': 'false',
 'new_id_artist': '6856a998-de4d-4469-ad72-a56c2334d283',
 'geo_id': '6d6106be-e4f6-4246-83f1-2c92cb6cc85f'}

In [21]:
for i in range(len(artists)):
    if not (
        artists[i].get('country') and
        artists[i].get('region') and
        artists[i].get('province')
    ):
        print(
            artists[i]['birth_place'], og_artists[i]['birth_place'],
            artists[i]['country'], og_artists[i]['country'],
            artists[i].get('region'), og_artists[i].get('region'),
            artists[i].get('province'), og_artists[i].get('province'),
            artists[i].get('latitude'), og_artists[i].get('latitude'),
            artists[i].get('longitude'), og_artists[i].get('longitude'),
        )
        print(artists[i])
        print(og_artists[i])

None None Italia None None None None None 42.6384261 None 12.6742970 None
{'id_author': 'ART18853907', 'name': 'alfa', 'gender': 'M', 'birth_date': None, 'birth_place': None, 'nationality': None, 'description': None, 'active_start': None, 'active_end': None, 'province': None, 'region': None, 'country': 'Italia', 'latitude': '42.6384261', 'longitude': '12.6742970', 'type': 'Person', 'active-end': 'false', 'new_id_artist': 'c3b079b6-faa0-4fae-85fd-2c91edf9fc89'}
{'id_author': 'ART18853907', 'name': 'alfa', 'gender': 'M', 'birth_date': None, 'birth_place': None, 'nationality': None, 'description': None, 'active_start': None, 'active_end': None, 'province': None, 'region': None, 'country': None, 'latitude': None, 'longitude': None}
None None None None None None None None None None None None
{'id_author': 'ART19605256', 'name': 'beba', 'gender': 'F', 'birth_date': None, 'birth_place': None, 'nationality': None, 'description': 'cognome', 'active_start': None, 'active_end': None, 'province': 

In [19]:
with open(f'{dataset_dir}/correct_ids/artists_correct_geo.json', 'w') as f:
    json.dump(artists, f)

with open(f'{dataset_dir}/correct_ids/geo.json', 'w') as f:
    json.dump(geo, f)